# Notebook 3: Wildfire Susceptibility Map

**Wildfire Susceptibility Mapping — Muğla Province, Turkey**  
CME434, Karabük University

**Purpose:** Apply best model to all pixels in Muğla, produce susceptibility raster  
**Input:** `best_model.pkl`, 15 feature rasters  
**Output:** `wildfire_susceptibility_map.tif`, `wildfire_susceptibility_map.png`

## Step 1 — Mount Drive & Install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/GIS_Wildfire_Mugla'

!pip install rasterio --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
import joblib
import os

print('Libraries loaded')

## Step 2 — Load Best Model

In [ ]:
model = joblib.load(f'{DRIVE}/best_model.pkl')
model_type = type(model).__name__
print(f'Model loaded: {model_type}')

scaler = None
scaler_path = f'{DRIVE}/scaler.pkl'
if os.path.exists(scaler_path):
    scaler = joblib.load(scaler_path)
    print('Scaler loaded')

uses_scaling = 'SVC' in model_type
print(f'Feature scaling required: {uses_scaling}')

## Step 3 — Load and Stack All 15 Feature Rasters

In [ ]:
FEATURE_FILES = [
    '01_elevation.tif',
    '02_slope.tif',
    '03_aspect.tif',
    '04_hillshade.tif',
    '05_tpi.tif',
    '06_ndvi.tif',
    '07_ndwi.tif',
    '08_evi.tif',
    '09_nbr.tif',
    '10_wind_speed.tif',
    '11_lst.tif',
    '12_soil_moisture.tif',
    '13_dist_roads.tif',
    '14_dist_urban.tif',
    '15_rainfall.tif',
]

# Read reference grid from first file (01_elevation.tif)
with rasterio.open(f'{DRIVE}/{FEATURE_FILES[0]}') as ref:
    profile   = ref.profile.copy()
    height    = ref.height
    width     = ref.width
    transform = ref.transform
    crs       = ref.crs
    nodata    = ref.nodata

print(f'Reference grid: {height} x {width} pixels')
print(f'CRS: {crs}  |  nodata: {nodata}')

# Stack all rasters, resampling to reference grid if shapes differ
bands = []
for fname in FEATURE_FILES:
    try:
        with rasterio.open(f'{DRIVE}/{fname}') as src:
            arr = src.read(1).astype(np.float32)
            if arr.shape != (height, width):
                resampled = np.empty((height, width), dtype=np.float32)
                reproject(
                    source=arr,
                    destination=resampled,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=crs,
                    resampling=Resampling.bilinear,
                )
                arr = resampled
                print(f'  Resampled {fname} → {height}x{width}')
        bands.append(arr)
        print(f'  Loaded {fname}  min={np.nanmin(arr):.3f}  max={np.nanmax(arr):.3f}')
    except Exception as e:
        print(f'  SKIPPED {fname} — error: {e}')

stack = np.stack(bands, axis=0)  # shape: (N_loaded, H, W)
print(f'\nStacked array shape: {stack.shape}')

## Step 4 — Build Valid Pixel Mask and Flatten

In [ ]:
# Mask pixels where any band is nodata or NaN
nd_val = nodata if nodata is not None else -9999
valid_mask = np.all((stack != nd_val) & np.isfinite(stack), axis=0)  # (H, W)

n_valid = valid_mask.sum()
n_total = height * width
print(f'Valid pixels: {n_valid:,} / {n_total:,} ({100*n_valid/n_total:.1f}%)')

# Reshape to (N_valid, 15)
X_flat = stack[:, valid_mask].T  # (N_valid, 15)
print(f'Feature matrix shape: {X_flat.shape}')

## Step 5 — Predict Susceptibility for Every Pixel

In [ ]:
if uses_scaling and scaler is not None:
    X_pred = scaler.transform(X_flat)
    print('Applying StandardScaler')
else:
    X_pred = X_flat

# Predict in chunks to avoid memory issues
CHUNK = 100_000
probs = []
for i in range(0, len(X_pred), CHUNK):
    chunk = X_pred[i:i+CHUNK]
    p = model.predict_proba(chunk)[:, 1]
    probs.append(p)
    print(f'  Processed {min(i+CHUNK, len(X_pred)):,} / {len(X_pred):,} pixels')

prob_flat = np.concatenate(probs)
print(f'\nPrediction range: {prob_flat.min():.3f} — {prob_flat.max():.3f}')
print(f'Mean susceptibility: {prob_flat.mean():.3f}')

## Step 6 — Reconstruct 2D Raster

In [ ]:
result = np.full((height, width), np.nan, dtype=np.float32)
result[valid_mask] = prob_flat
print(f'Output raster shape: {result.shape}')

## Step 7 — Save wildfire_susceptibility_map.tif

In [ ]:
out_path = f'{DRIVE}/wildfire_susceptibility_map.tif'

out_profile = profile.copy()
out_profile.update(dtype='float32', count=1, nodata=np.nan)

with rasterio.open(out_path, 'w', **out_profile) as dst:
    dst.write(result, 1)

size_mb = os.path.getsize(out_path) / 1e6
print(f'Saved: {out_path}  ({size_mb:.1f} MB)')

## Step 8 — Plot Susceptibility Map

In [ ]:
# Red-yellow-green colormap (high=red, low=green)
cmap = LinearSegmentedColormap.from_list(
    'wildfire', ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'])

fig, ax = plt.subplots(figsize=(11, 9))
img = ax.imshow(result, cmap=cmap, vmin=0, vmax=1,
                extent=[transform.c,
                        transform.c + width  * transform.a,
                        transform.f + height * transform.e,
                        transform.f])

cbar = plt.colorbar(img, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Wildfire Susceptibility Probability', fontsize=10)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(['Very Low', 'Low', 'Medium', 'High', 'Very High'])

ax.set_title(
    'Muğla Province — Wildfire Susceptibility Map\n'
    'CME434 | Karabük University | 2025–2026',
    fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.grid(alpha=0.25, linestyle='--', linewidth=0.5)

plt.tight_layout()
fig_path = f'{DRIVE}/wildfire_susceptibility_map.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Step 9 — Summary Statistics by Risk Class

In [ ]:
flat = result[valid_mask]
breaks = [0.0, 0.20, 0.40, 0.60, 0.80, 1.01]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

print('Risk class distribution (% of valid pixels):')
for i, label in enumerate(labels):
    lo, hi = breaks[i], breaks[i+1]
    pct = 100 * ((flat >= lo) & (flat < hi)).sum() / len(flat)
    print(f'  {label:<12}: {pct:5.1f}%')

print(f'\nHigh + Very High: {100*((flat >= 0.60).sum()/len(flat)):.1f}% of Muğla')
print('\nDone.')